# 35. Nested LOPO RRF Optimization for HS4

실험 34의 데이터·평가 집단을 유지하면서, 바깥쪽 LOPO 평가 Fold 안에서 안쪽 LOPO를 수행해 RRF 결합 설정을 선택합니다. 주 지표는 HS4 Macro-F1, 보조 지표는 HS4 Accuracy입니다. 바깥쪽 평가 Fold는 하이퍼파라미터 선택에 사용하지 않습니다. Google Drive는 사용하지 않습니다.

## 실험 설계

기준 모델은 실험 34의 SBERT Product + Ruling RRF(top-3 mean, w=1)입니다. 개선 모델은 바깥쪽 학습 데이터에서 안쪽 LOPO를 수행해 Product/Ruling 집계 방식과 RRF 가중치를 선택합니다. 집계 방식은 max·mean·top-3 mean, RRF 가중치는 0.25·0.5·1·2·4를 탐색합니다. 선택된 설정으로 바깥쪽 홀드아웃 제품군을 예측합니다. 모든 전처리·검색·튜닝은 각 Fold의 학습 데이터 안에서 수행합니다.

## 0. 환경 설정

In [ ]:
!pip -q install -U sentence-transformers kiwipiepy scipy
import re, json, zipfile, requests
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
WORK=Path('/content/35_hs4_nested_rrf_results')
WORK.mkdir(parents=True,exist_ok=True)
print(WORK)

## 1. 29번과 동일한 데이터 파이프라인

In [ ]:
BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/'
src=requests.get(BASE+'29_discriminative_features_experiment.ipynb').json() if 'requests' in globals() else None
if src is None:
 import requests
 src=requests.get(BASE+'29_discriminative_features_experiment.ipynb').json()
for i in [3,5,7]: exec(''.join(src['cells'][i]['source']),globals())
print('평가 표본:',len(eval_df),'| HS6:',len(labels),'| 제품군:',eval_df.product_family.nunique(),'| HS4:',eval_df.hs4.nunique())
assert len(eval_df)==195 and len(labels)==11 and eval_df.product_family.nunique()==29
hs4_labels=sorted(eval_df.hs4.astype(str).unique())
print('HS4 labels:',hs4_labels)
print(eval_df.groupby('hs4').agg(n=('hs4','size'),families=('product_family','nunique')).to_string())

## 2. 임베딩 생성 및 캐시

In [ ]:
CACHE=WORK/'embeddings_34.npz'
if CACHE.exists():
 z=np.load(CACHE,allow_pickle=True); query_emb,product_emb,ruling_emb=z['query_emb'],z['product_emb'],z['ruling_emb']
else:
 sbert=SentenceTransformer('intfloat/multilingual-e5-base')
 query_emb=sbert.encode(['query: '+str(x) for x in eval_df.query_text],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
 product_emb=sbert.encode(['passage: '+str(x) for x in eval_df.query_text],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
 ruling_emb=sbert.encode(['passage: '+str(x) for x in eval_df.ruling_text],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
 np.savez_compressed(CACHE,query_emb=query_emb,product_emb=product_emb,ruling_emb=ruling_emb)
print(query_emb.shape,product_emb.shape,ruling_emb.shape)

## 3. HS4 점수 함수 및 Fold

In [ ]:
FOLD_START=1
FOLD_END=5
METHODS=['max','mean','top3_mean']
WEIGHTS=[0.25,0.5,1.0,2.0,4.0]
def get_rank_pair(train_idx,query_idx):
    tr_hs4=eval_df.iloc[train_idx].hs4.astype(str).to_numpy()
    sp=product_emb[train_idx]@query_emb[query_idx]
    sr=ruling_emb[train_idx]@query_emb[query_idx]
    return {m:aggregate_hs4(sp,tr_hs4,m) for m in METHODS},{m:aggregate_hs4(sr,tr_hs4,m) for m in METHODS}
def predict_rrf(rp,rr,mp,mr,w):
    return str(rrf_hs4(rp[mp],rr[mr],weight_b=w).index[0])
def tune_inner(train_idx):
    inner_df=eval_df.iloc[train_idx].reset_index(drop=True)
    inner_groups=inner_df.product_family.to_numpy()
    inner_splits=list(LeaveOneGroupOut().split(inner_df,inner_df.hs4,groups=inner_groups))
    candidates=[(mp,mr,w) for mp in METHODS for mr in METHODS for w in WEIGHTS]
    records={c:[] for c in candidates}
    for itr,iva in inner_splits:
        global_tr=train_idx[itr]
        global_va=train_idx[iva]
        for local_j,global_j in enumerate(global_va):
            rp,rr=get_rank_pair(global_tr,global_j)
            true=str(eval_df.iloc[global_j].hs4)
            for c in candidates:
                mp,mr,w=c
                records[c].append((true,predict_rrf(rp,rr,mp,mr,w)))
    scored=[]
    labels=sorted(eval_df.iloc[train_idx].hs4.astype(str).unique())
    for c,vals in records.items():
        yt=[v[0] for v in vals]; yp=[v[1] for v in vals]
        mf=f1_score(yt,yp,labels=labels,average='macro',zero_division=0)
        acc=accuracy_score(yt,yp)
        scored.append((mf,acc,c))
    scored.sort(key=lambda x:(x[0],x[1]),reverse=True)
    return scored[0][2],scored[0][0],scored[0][1]
for fold,(tr,te) in enumerate(splits,1):
    if not FOLD_START<=fold<=FOLD_END: continue
    if result_path(fold).exists(): print('Fold',fold,'완료 - skip'); continue
    train_hs4=eval_df.iloc[tr].hs4.astype(str).to_numpy()
    test_hs4=eval_df.iloc[te].hs4.astype(str).to_numpy()
    if not set(test_hs4).issubset(set(train_hs4)): raise ValueError(f'Fold {fold}: 학습 데이터에 없는 HS4')
    best_cfg,inner_f1,inner_acc=tune_inner(tr)
    mp,mr,w=best_cfg
    rows=[]
    for global_i in te:
        rp,rr=get_rank_pair(tr,global_i)
        baseline=predict_rrf(rp,rr,'top3_mean','top3_mean',1.0)
        tuned=predict_rrf(rp,rr,mp,mr,w)
        common={'row_index':int(global_i),'fold':fold,'heldout_family':eval_df.iloc[global_i].product_family,'true_hs4':str(eval_df.iloc[global_i].hs4)}
        rows.append({**common,'model':'Baseline_RRF_top3_w1','predicted_hs4':baseline,'selected_product_method':'top3_mean','selected_ruling_method':'top3_mean','selected_weight':1.0,'inner_macro_f1':np.nan,'inner_accuracy':np.nan})
        rows.append({**common,'model':'Nested_LOPO_Tuned_RRF','predicted_hs4':tuned,'selected_product_method':mp,'selected_ruling_method':mr,'selected_weight':w,'inner_macro_f1':inner_f1,'inner_accuracy':inner_acc})
    pd.DataFrame(rows).to_csv(result_path(fold),index=False,encoding='utf-8-sig')
    print(f'Fold {fold}/{len(splits)} 저장 | 선택 설정: Product={mp}, Ruling={mr}, w={w} | inner Macro-F1={inner_f1:.4f}')
print('실행 구간 완료')

## 4. Fold별 HS4 직접 분류

각 Fold에서 TF-IDF 모델은 학습 Fold에만 적합합니다. SBERT 검색도 학습 Fold 사례만 검색합니다. 실행은 5개 Fold 단위이며 완료 Fold는 자동으로 건너뜁니다.

In [ ]:
FOLD_START=1
FOLD_END=5
C_VALUES=[0.3,1.0,3.0]
for fold,(tr,te) in enumerate(splits,1):
 if not FOLD_START<=fold<=FOLD_END: continue
 if result_path(fold).exists(): print('Fold',fold,'완료 - skip'); continue
 train=eval_df.iloc[tr].copy(); train_hs4=train.hs4.astype(str).to_numpy(); train_hs6=train.hs6.astype(str).to_numpy()
 if not set(eval_df.iloc[te].hs4.astype(str)).issubset(set(train_hs4)): raise ValueError(f'Fold {fold}: 학습 데이터에 없는 HS4')
 rows=[]
 vectorizer=FeatureUnion([('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1,max_features=30000,sublinear_tf=True)),('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=30000,sublinear_tf=True))])
 x_train=vectorizer.fit_transform(train.model_text); x_test=vectorizer.transform(eval_df.iloc[te].model_text)
 for c in C_VALUES:
  clf=LinearSVC(C=c,class_weight='balanced').fit(x_train,train_hs4)
  dec=clf.decision_function(x_test)
  if dec.ndim==1: dec=np.column_stack([-dec,dec])
  for j,global_i in enumerate(te):
   rank=pd.Series(dec[j],index=clf.classes_).sort_values(ascending=False)
   rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':eval_df.iloc[global_i].product_family,'true_hs4':str(eval_df.iloc[global_i].hs4),'model':f'TFIDF_LinearSVC_balanced_C{c:g}','predicted_hs4':str(rank.index[0]),'ranked_hs4':'|'.join(map(str,rank.index))})
 for global_i in te:
  scores_product=product_emb[tr]@query_emb[global_i]
  scores_ruling=ruling_emb[tr]@query_emb[global_i]
  for method in ['max','mean','top3_mean']:
   rp=aggregate_hs4(scores_product,train_hs4,method)
   rr=aggregate_hs4(scores_ruling,train_hs4,method)
   for name,rank in [(f'SBERT_product_{method}',rp),(f'SBERT_ruling_{method}',rr)]:
    rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':eval_df.iloc[global_i].product_family,'true_hs4':str(eval_df.iloc[global_i].hs4),'model':name,'predicted_hs4':str(rank.index[0]),'ranked_hs4':'|'.join(map(str,rank.index))})
  rp=aggregate_hs4(scores_product,train_hs4,'top3_mean'); rr=aggregate_hs4(scores_ruling,train_hs4,'top3_mean')
  for weight in [0.5,1.0,2.0]:
   rank=rrf_hs4(rp,rr,weight_b=weight)
   rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':eval_df.iloc[global_i].product_family,'true_hs4':str(eval_df.iloc[global_i].hs4),'model':f'SBERT_product_ruling_RRF_w{weight:g}','predicted_hs4':str(rank.index[0]),'ranked_hs4':'|'.join(map(str,rank.index))})
 pd.DataFrame(rows).to_csv(result_path(fold),index=False,encoding='utf-8-sig')
 print(f'Fold {fold}/{len(splits)} 저장 완료 - {len(te)}건')
print('실행 구간 완료')

## 5. 진행률 및 중간 백업

In [ ]:
done=sorted(int(p.stem.split('_')[1]) for p in WORK.glob('fold_*.csv')); missing=sorted(set(range(1,len(splits)+1))-set(done))
print(f'완료 Fold: {len(done)}/{len(splits)}'); print('누락:',missing)
BACKUP='/content/34_fold_backup.zip'
with zipfile.ZipFile(BACKUP,'w',zipfile.ZIP_DEFLATED) as z:
 for p in WORK.glob('fold_*.csv'): z.write(p,p.name)
from google.colab import files
files.download(BACKUP)

## 6. 중간 백업 복구 (새 런타임에서만 실행)

In [ ]:
from google.colab import files
uploaded=files.upload()
for name in uploaded:
 if name.endswith('.zip'):
  with zipfile.ZipFile(name) as z: z.extractall(WORK)
  print('복구 완료:',name)

## 7. 통합 평가 — HS4 Macro-F1 우선

In [ ]:
files_list=sorted(WORK.glob('fold_*.csv'))
done=sorted(int(p.stem.split('_')[1]) for p in files_list); missing=sorted(set(range(1,len(splits)+1))-set(done))
print(f'완료 Fold: {len(done)}/{len(splits)} | 누락: {missing}')
if missing: raise ValueError('미완료 Fold가 있습니다.')
pred=pd.concat([pd.read_csv(p) for p in files_list],ignore_index=True)
if pred.duplicated(['row_index','model']).any(): raise ValueError('중복 예측 존재')
def evaluate(g):
 yt=g.true_hs4.astype(str); yp=g.predicted_hs4.astype(str)
 return pd.Series({'n':len(g),'HS4_Macro_F1':f1_score(yt,yp,labels=hs4_labels,average='macro',zero_division=0),'HS4_Accuracy':accuracy_score(yt,yp)})
summary=pred.groupby('model').apply(evaluate,include_groups=False).reset_index().sort_values(['HS4_Macro_F1','HS4_Accuracy'],ascending=False)
class_rows=[]
for model,g in pred.groupby('model'):
 p,r,f,s=precision_recall_fscore_support(g.true_hs4.astype(str),g.predicted_hs4.astype(str),labels=hs4_labels,zero_division=0)
 for c,pv,rv,fv,sv in zip(hs4_labels,p,r,f,s): class_rows.append({'model':model,'HS4':c,'support':int(sv),'precision':pv,'recall':rv,'F1':fv})
class_metrics=pd.DataFrame(class_rows)
display(summary.round(4)); display(class_metrics.sort_values(['model','HS4']).round(4))
OUT=WORK/'final'; OUT.mkdir(exist_ok=True)
pred.to_csv(OUT/'34_oof_predictions.csv',index=False,encoding='utf-8-sig'); summary.to_csv(OUT/'34_model_comparison.csv',index=False,encoding='utf-8-sig'); class_metrics.to_csv(OUT/'34_hs4_class_metrics.csv',index=False,encoding='utf-8-sig')
for model,g in pred.groupby('model'):
 cm=pd.DataFrame(confusion_matrix(g.true_hs4.astype(str),g.predicted_hs4.astype(str),labels=hs4_labels),index=hs4_labels,columns=hs4_labels)
 cm.to_csv(OUT/f'confusion_{model}.csv',encoding='utf-8-sig')
ZIP='/content/34_hs4_direct_results.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
 for p in OUT.glob('*.csv'): z.write(p,p.name)
print('최종 ZIP:',ZIP)

## 8. 최종 결과 PC 다운로드

In [ ]:
from google.colab import files
files.download('/content/34_hs4_direct_results.zip')